# 01_market_prices_silver

## Purpose

Transform Bronze market price records into a cleaned and reusable Silver table.

This notebook standardizes text fields, casts numeric fields, adds a reporting day, filters invalid records, and writes the result to the Silver layer.

In [0]:
import sys

repo_src_path = "/Workspace/Users/dirella@startsteps.org/vattenfall-week9-capstone-anna/src"

if repo_src_path not in sys.path:
    sys.path.append(repo_src_path)

from transforms.market_price_transforms import (
    standardize_market_price_columns,
    cast_market_price_fields,
    add_market_price_day,
    filter_invalid_market_prices,
)

from validation.silver_checks import print_basic_quality_summary

In [0]:
catalog = "vattenfall_dev"

bronze_table = f"{catalog}.raw.bronze_market_prices"
silver_table = f"{catalog}.refined.silver_market_prices"

print("Source Bronze table:", bronze_table)
print("Target Silver table:", silver_table)

In [0]:
bronze_market_df = spark.table(bronze_table)

before_count = bronze_market_df.count()

print("Rows before cleaning:", before_count)
display(bronze_market_df.limit(20))

In [0]:
silver_market_df = (
    bronze_market_df
    .transform(standardize_market_price_columns)
    .transform(cast_market_price_fields)
    .transform(add_market_price_day)
    .transform(filter_invalid_market_prices)
)

after_count = silver_market_df.count()

print("Rows after cleaning:", after_count)
print("Rows removed:", before_count - after_count)

display(silver_market_df.limit(20))

In [0]:
(
    silver_market_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(silver_table)
)

print("Written Silver table:", silver_table)

In [0]:
silver_df = spark.table(silver_table)

print_basic_quality_summary(silver_df, silver_table)

display(silver_df.limit(20))

In [0]:
print("Market type values:")
silver_df.select("market_type").distinct().show(truncate=False)

print("Null check for key Silver fields:")
for column_name in ["region", "market_type", "price_eur_mwh", "volume_mwh", "report_day"]:
    null_count = silver_df.filter(f"{column_name} IS NULL").count()
    print(column_name, "nulls:", null_count)